In [2]:
import os
import requests
import zipfile
from datetime import datetime
import pandas as pd

# 1. Create Medallion folders
for folder in ['bronze', 'silver', 'gold']:
    os.makedirs(folder, exist_ok=True)

# 2. Download and unzip the dataset (using a browser User-Agent to avoid 403)
url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"
zip_path = "book-crossings.zip"

if not os.path.exists(zip_path):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    with open(zip_path, "wb") as f:
        f.write(response.content)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("raw_data")

# 3. Load raw CSVs (semicolon-separated, latin-1 encoding)
users_raw = pd.read_csv("raw_data/BX-Users.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
books_raw = pd.read_csv("raw_data/BX-Books.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
ratings_raw = pd.read_csv("raw_data/BX-Book-Ratings.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)

# 4. Add ingestion_timestamp and save to bronze/ as Parquet (no cleaning)
ingestion_time = datetime.utcnow().isoformat()

for name, df in [("users", users_raw), ("books", books_raw), ("ratings", ratings_raw)]:
    df["ingestion_timestamp"] = ingestion_time
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str)
    df.to_parquet(f"bronze/{name}.parquet", index=False)

print("Bronze layer saved successfully!")

/tmp/ipykernel_708/489888135.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingestion_time = datetime.utcnow().isoformat()


Bronze layer saved successfully!


In [3]:
import pandas as pd
import numpy as np

users_b = pd.read_parquet("bronze/users.parquet")
books_b = pd.read_parquet("bronze/books.parquet")
ratings_b = pd.read_parquet("bronze/ratings.parquet")

# 1. Shape, data types, and missing values
for name, df in [("Users", users_b), ("Books", books_b), ("Ratings", ratings_b)]:
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    print(df.dtypes)
    # Count nulls (including 'nan' strings created during raw Parquet cast)
    nulls = df.replace("nan", np.nan).isna().sum()
    print("Missing values:\n", nulls[nulls > 0], "\n")

# 2. Distribution of ratings (especially 0s)
print("=== Rating Distribution ===")
print(ratings_b["Book-Rating"].value_counts().sort_index())
zero_pct = (ratings_b["Book-Rating"] == 0).mean() * 100
print(f"Zero ratings: {(ratings_b['Book-Rating'] == 0).sum()} ({zero_pct:.2f}%)\n")

# 3. Distribution of user ages and publication years
ages = pd.to_numeric(users_b["Age"], errors="coerce")
print("=== User Ages ===")
print(ages.describe())
print(f"Ages < 5 or > 100: {((ages < 5) | (ages > 100)).sum()}\n")

years = pd.to_numeric(books_b["Year-Of-Publication"], errors="coerce")
print("=== Publication Years ===")
print(years.describe())
print(f"Years == 0 or > 2026: {((years == 0) | (years > 2026)).sum()}\n")

# 4. Ratings per user and per book
user_counts = ratings_b["User-ID"].value_counts()
book_counts = ratings_b["ISBN"].value_counts()
print("=== Ratings per User ===\n", user_counts.describe())
print("\n=== Ratings per Book ===\n", book_counts.describe())

# 5. Matrix sparsity
n_users = ratings_b["User-ID"].nunique()
n_books = ratings_b["ISBN"].nunique()
n_ratings = len(ratings_b)
density = (n_ratings / (n_users * n_books)) * 100
print(f"\n=== Matrix Sparsity ===")
print(f"Unique Users: {n_users:,}, Unique Books: {n_books:,}")
print(f"Density: {density:.6f}% | Sparsity: {100 - density:.6f}%")

=== Users ===
Shape: (278858, 4)
User-ID                  int64
Location                object
Age                    float64
ingestion_timestamp     object
dtype: object
Missing values:
 Age    110762
dtype: int64 

=== Books ===
Shape: (271360, 9)
ISBN                   object
Book-Title             object
Book-Author            object
Year-Of-Publication    object
Publisher              object
Image-URL-S            object
Image-URL-M            object
Image-URL-L            object
ingestion_timestamp    object
dtype: object
Missing values:
 Book-Author    2
Publisher      2
Image-URL-L    3
dtype: int64 

=== Ratings ===
Shape: (1149780, 4)
User-ID                 int64
ISBN                   object
Book-Rating             int64
ingestion_timestamp    object
dtype: object
Missing values:
 Series([], dtype: int64) 

=== Rating Distribution ===
Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541

### Step 2: Bronze EDA Findings & Silver Plan
* **Shape, data types, and missing values:** `Users` (278,858 rows) has 110,762 missing `Age` values, `Books` (~271,360 rows) has 2 missing authors, 2 missing publishers, and 3 missing `Image-URL-L` values, and `Ratings` (1,149,780 rows) has no missing values; in Silver, we will drop the unneeded image-URL columns, clean text formatting, and handle missing values.
* **Distribution of ratings:** 716,109 ratings (62.28%) are `0` (implicit/no explicit rating); in Silver, we will remove all `0` ratings and keep only explicit ratings from 1 to 10.
* **User ages and publication years:** There are 1,248 invalid ages (< 5 or > 100, ranging from 0 to 244) and 4,629 invalid publication years (0 or > 2026, up to 2050); in Silver, we will set invalid ages to null and fill all null ages with the median (32), and set invalid publication years to null.
* **Ratings per user and per book:** The data is heavily skewed—the median user and median book each have only 1 rating (75% of users have ≤ 4 ratings and 75% of books have ≤ 2); in Silver, we will filter to keep only users and books with at least 10 explicit ratings so recommendations are reliable.
* **Matrix sparsity:** Across 105,283 unique users and 340,556 unique books, matrix density is only 0.003207% (99.996793% sparse); in Gold, we will store the user-book matrix as a `scipy.sparse` CSR matrix.

In [4]:
import pandas as pd
import numpy as np

users = pd.read_parquet("bronze/users.parquet").replace("nan", np.nan)
books = pd.read_parquet("bronze/books.parquet").replace("nan", np.nan)
ratings = pd.read_parquet("bronze/ratings.parquet").replace("nan", np.nan)

# 1. Remove duplicates
users = users.drop_duplicates(subset=["User-ID"])
books = books.drop_duplicates(subset=["ISBN"])
ratings = ratings.drop_duplicates(subset=["User-ID", "ISBN"])

# 2. Remove ratings of 0; keep 1 to 10
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce")
ratings = ratings[ratings["Book-Rating"].between(1, 10)].copy()

# 3. Fix user ages (< 5 or > 100 -> null, then fill with median)
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
users.loc[(users["Age"] < 5) | (users["Age"] > 100), "Age"] = np.nan
median_age = users["Age"].median()
users["Age"] = users["Age"].fillna(median_age)

# 4. Split Location into city, state, country
loc_split = users["Location"].fillna("").str.split(",", n=2, expand=True)
users["city"] = loc_split[0].str.strip().str.lower()
users["state"] = loc_split[1].str.strip().str.lower() if loc_split.shape[1] > 1 else ""
users["country"] = loc_split[2].str.strip().str.lower() if loc_split.shape[1] > 2 else ""
users = users.drop(columns=["Location"])

# 5. Fix publication years (0 or future > 2026 -> null)
books["Year-Of-Publication"] = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
books.loc[(books["Year-Of-Publication"] == 0) | (books["Year-Of-Publication"] > 2026), "Year-Of-Publication"] = np.nan

# 6. Clean text columns and drop image-URL columns
url_cols = [c for c in books.columns if "Image-URL" in c]
books = books.drop(columns=url_cols)

for col in ["Book-Title", "Book-Author", "Publisher"]:
    books[col] = books[col].fillna("").str.strip().str.title()

books["ISBN"] = books["ISBN"].str.strip()
ratings["ISBN"] = ratings["ISBN"].str.strip()

# 7. Keep only ratings whose book and user exist in books and users tables
ratings = ratings[ratings["ISBN"].isin(books["ISBN"]) & ratings["User-ID"].isin(users["User-ID"])]

# 8. Filter by minimum number of ratings
# Threshold justification: EDA showed >99.99% sparsity with most users/books having 1-2 ratings.
# Keeping users and books with >= 10 explicit ratings removes noise and produces reliable cosine similarities.
MIN_USER_RATINGS = 10
MIN_BOOK_RATINGS = 10

active_users = ratings["User-ID"].value_counts()
active_users = active_users[active_users >= MIN_USER_RATINGS].index

popular_books = ratings["ISBN"].value_counts()
popular_books = popular_books[popular_books >= MIN_BOOK_RATINGS].index

ratings_silver = ratings[ratings["User-ID"].isin(active_users) & ratings["ISBN"].isin(popular_books)].copy()
users_silver = users[users["User-ID"].isin(ratings_silver["User-ID"])].copy()
books_silver = books[books["ISBN"].isin(ratings_silver["ISBN"])].copy()

# Save to silver/
users_silver.to_parquet("silver/users.parquet", index=False)
books_silver.to_parquet("silver/books.parquet", index=False)
ratings_silver.to_parquet("silver/ratings.parquet", index=False)

print(f"Silver Layer Saved -> Users: {len(users_silver):,}, Books: {len(books_silver):,}, Ratings: {len(ratings_silver):,}")

Silver Layer Saved -> Users: 6,318, Books: 5,438, Ratings: 87,178


In [5]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz

ratings_s = pd.read_parquet("silver/ratings.parquet")
books_s = pd.read_parquet("silver/books.parquet")

# 1. Create integer indices for users and books
user_ids = ratings_s["User-ID"].unique()
book_ids = ratings_s["ISBN"].unique()

user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
book_to_idx = {isbn: i for i, isbn in enumerate(book_ids)}

# Save mappings for Gold layer
pd.DataFrame({"User-ID": user_ids, "user_idx": range(len(user_ids))}).to_parquet("gold/user_mapping.parquet", index=False)
pd.DataFrame({"ISBN": book_ids, "book_idx": range(len(book_ids))}).to_parquet("gold/book_mapping.parquet", index=False)

# 2. Build and save sparse User-Book rating matrix (rows=users, cols=books)
row_idx = ratings_s["User-ID"].map(user_to_idx).values
col_idx = ratings_s["ISBN"].map(book_to_idx).values
vals = ratings_s["Book-Rating"].values.astype(np.float32)

user_book_matrix = csr_matrix((vals, (row_idx, col_idx)), shape=(len(user_ids), len(book_ids)))
save_npz("gold/user_book_matrix.npz", user_book_matrix)

# 3. Book lookup table (ISBN, title, author)
book_lookup = books_s[["ISBN", "Book-Title", "Book-Author"]].drop_duplicates(subset=["ISBN"])
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

# 4. User-level and Book-level aggregates
user_agg = ratings_s.groupby("User-ID")["Book-Rating"].agg(avg_rating="mean", num_ratings="count").reset_index()
book_agg = ratings_s.groupby("ISBN")["Book-Rating"].agg(avg_rating="mean", num_ratings="count").reset_index()

user_agg.to_parquet("gold/user_aggregates.parquet", index=False)
book_agg.to_parquet("gold/book_aggregates.parquet", index=False)

print("Gold layer built! Sparse matrix shape (Users x Books):", user_book_matrix.shape)

Gold layer built! Sparse matrix shape (Users x Books): (6318, 5438)


In [6]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

# 1. Load Gold data
user_book_matrix = load_npz("gold/user_book_matrix.npz")
book_user_matrix = user_book_matrix.T.tocsr()  # Rows = Books, Cols = Users

user_map = pd.read_parquet("gold/user_mapping.parquet")
book_map = pd.read_parquet("gold/book_mapping.parquet")
book_lookup = pd.read_parquet("gold/book_lookup.parquet")

# Fast lookup dictionaries
uid_to_idx = dict(zip(user_map["User-ID"], user_map["user_idx"]))
idx_to_isbn = dict(zip(book_map["book_idx"], book_map["ISBN"]))
isbn_to_idx = dict(zip(book_map["ISBN"], book_map["book_idx"]))
isbn_to_info = book_lookup.set_index("ISBN")[["Book-Title", "Book-Author"]].to_dict("index")

# 2. Fit Item-Based Collaborative Filtering model (Cosine Similarity)
model = NearestNeighbors(metric="cosine", algorithm="brute")
model.fit(book_user_matrix)

# 3. Function: Given a book title, return 10 most similar books
def get_similar_books(title, n=10):
    matches = book_lookup[book_lookup["Book-Title"].str.contains(title, case=False, na=False)]
    if matches.empty:
        return f"No book found matching '{title}'"

    isbn = matches.iloc[0]["ISBN"]
    b_idx = isbn_to_idx[isbn]

    distances, indices = model.kneighbors(book_user_matrix[b_idx], n_neighbors=n + 1)

    results = []
    for dist, idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
        sim_isbn = idx_to_isbn[idx]
        info = isbn_to_info[sim_isbn]
        results.append({
            "Title": info["Book-Title"],
            "Author": info["Book-Author"],
            "Similarity": round(1 - dist, 4)
        })
    return pd.DataFrame(results)

# 4. Function: recommend(user_id, n=10)
def recommend(user_id, n=10):
    if user_id not in uid_to_idx:
        return f"User {user_id} not found in Gold layer."

    u_idx = uid_to_idx[user_id]
    user_row = user_book_matrix[u_idx]

    # Books the user has already read
    read_indices = set(user_row.indices)
    user_ratings = dict(zip(user_row.indices, user_row.data))

    # Focus on books the user rated highly (>= 7, or their top-rated books if none >= 7)
    high_rated = [b_idx for b_idx, r in user_ratings.items() if r >= 7]
    if not high_rated:
        high_rated = sorted(user_ratings, key=user_ratings.get, reverse=True)[:5]

    # Collect similar books and score them by (similarity * user_rating)
    scores = {}
    for b_idx in high_rated:
        rating = user_ratings[b_idx]
        distances, indices = model.kneighbors(book_user_matrix[b_idx], n_neighbors=15)
        for dist, sim_idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
            if sim_idx not in read_indices:
                similarity = 1 - dist
                scores[sim_idx] = scores.get(sim_idx, 0) + (similarity * rating)

    # Rank and return top n
    top_indices = sorted(scores, key=scores.get, reverse=True)[:n]
    recs = []
    for idx in top_indices:
        isbn = idx_to_isbn[idx]
        info = isbn_to_info[isbn]
        recs.append({
            "Title": info["Book-Title"],
            "Author": info["Book-Author"],
            "Score": round(scores[idx], 3)
        })
    return pd.DataFrame(recs)

# Quick test of similar books
print("Books similar to 'Harry Potter':")
display(get_similar_books("Harry Potter", n=5))

Books similar to 'Harry Potter':


,Title,Author,Similarity
0,Jacob Have I Loved,Katherine Paterson,0.3488
1,Due Di Due (Bestsellers),Andrea De Carlo,0.3439
2,Free,Paul Vincent,0.3439
3,"Novocento, Un Monologo",Alessandro Baricco,0.3439
4,Oceano Mare,Alessandro Baricco,0.3439


In [7]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

# 1. Load Gold data
user_book_matrix = load_npz("gold/user_book_matrix.npz")
book_user_matrix = user_book_matrix.T.tocsr()  # Rows = Books, Cols = Users

user_map = pd.read_parquet("gold/user_mapping.parquet")
book_map = pd.read_parquet("gold/book_mapping.parquet")
book_lookup = pd.read_parquet("gold/book_lookup.parquet")

# Fast lookup dictionaries
uid_to_idx = dict(zip(user_map["User-ID"], user_map["user_idx"]))
idx_to_isbn = dict(zip(book_map["book_idx"], book_map["ISBN"]))
isbn_to_idx = dict(zip(book_map["ISBN"], book_map["book_idx"]))
isbn_to_info = book_lookup.set_index("ISBN")[["Book-Title", "Book-Author"]].to_dict("index")

# 2. Fit Item-Based Collaborative Filtering model (Cosine Similarity)
model = NearestNeighbors(metric="cosine", algorithm="brute")
model.fit(book_user_matrix)

# 3. Function: Given a book title, return 10 most similar books
def get_similar_books(title, n=10):
    matches = book_lookup[book_lookup["Book-Title"].str.contains(title, case=False, na=False)]
    if matches.empty:
        return f"No book found matching '{title}'"

    isbn = matches.iloc[0]["ISBN"]
    b_idx = isbn_to_idx[isbn]

    distances, indices = model.kneighbors(book_user_matrix[b_idx], n_neighbors=n + 1)

    results = []
    for dist, idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
        sim_isbn = idx_to_isbn[idx]
        info = isbn_to_info[sim_isbn]
        results.append({
            "Title": info["Book-Title"],
            "Author": info["Book-Author"],
            "Similarity": round(1 - dist, 4)
        })
    return pd.DataFrame(results)

# 4. Function: recommend(user_id, n=10)
def recommend(user_id, n=10):
    if user_id not in uid_to_idx:
        return f"User {user_id} not found in Gold layer."

    u_idx = uid_to_idx[user_id]
    user_row = user_book_matrix[u_idx]

    # Books the user has already read
    read_indices = set(user_row.indices)
    user_ratings = dict(zip(user_row.indices, user_row.data))

    # Focus on books the user rated highly (>= 7, or their top-rated books if none >= 7)
    high_rated = [b_idx for b_idx, r in user_ratings.items() if r >= 7]
    if not high_rated:
        high_rated = sorted(user_ratings, key=user_ratings.get, reverse=True)[:5]

    # Collect similar books and score them by (similarity * user_rating)
    scores = {}
    for b_idx in high_rated:
        rating = user_ratings[b_idx]
        distances, indices = model.kneighbors(book_user_matrix[b_idx], n_neighbors=15)
        for dist, sim_idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
            if sim_idx not in read_indices:
                similarity = 1 - dist
                scores[sim_idx] = scores.get(sim_idx, 0) + (similarity * rating)

    # Rank and return top n
    top_indices = sorted(scores, key=scores.get, reverse=True)[:n]
    recs = []
    for idx in top_indices:
        isbn = idx_to_isbn[idx]
        info = isbn_to_info[isbn]
        recs.append({
            "Title": info["Book-Title"],
            "Author": info["Book-Author"],
            "Score": round(scores[idx], 3)
        })
    return pd.DataFrame(recs)

# Quick test of similar books
print("Books similar to 'Harry Potter':")
display(get_similar_books("Harry Potter", n=5))

Books similar to 'Harry Potter':


,Title,Author,Similarity
0,Jacob Have I Loved,Katherine Paterson,0.3488
1,Due Di Due (Bestsellers),Andrea De Carlo,0.3439
2,Free,Paul Vincent,0.3439
3,"Novocento, Un Monologo",Alessandro Baricco,0.3439
4,Oceano Mare,Alessandro Baricco,0.3439


In [8]:
# Pick 3 active users from the Gold layer
user_agg = pd.read_parquet("gold/user_aggregates.parquet")
sample_users = user_agg.sort_values("num_ratings", ascending=False)["User-ID"].iloc[5:8].tolist()

for uid in sample_users:
    print("=" * 70)
    print(f"USER ID: {uid}")

    # Get user's top rated books
    u_idx = uid_to_idx[uid]
    u_row = user_book_matrix[u_idx]
    rated_books = [
        (isbn_to_info[idx_to_isbn[b_idx]]["Book-Title"], rating)
        for b_idx, rating in zip(u_row.indices, u_row.data)
    ]
    rated_books = sorted(rated_books, key=lambda x: x[1], reverse=True)[:5]

    print("\nTop Books Rated Highly by User:")
    for title, r in rated_books:
        print(f"  - [{int(r)}/10] {title}")

    print("\nTop 10 Recommended Books:")
    display(recommend(uid, n=10))

USER ID: 104636

Top Books Rated Highly by User:
  - [10/10] A Time To Kill
  - [10/10] Where The Heart Is (Oprah'S Book Club (Paperback))
  - [10/10] Harry Potter And The Chamber Of Secrets (Book 2)
  - [10/10] Harry Potter And The Prisoner Of Azkaban (Book 3)
  - [10/10] Harry Potter And The Goblet Of Fire (Book 4)

Top 10 Recommended Books:


,Title,Author,Score
0,Oceano Mare,Alessandro Baricco,89.263000
1,"Novocento, Un Monologo",Alessandro Baricco,85.365997
2,Due Di Due (Bestsellers),Andrea De Carlo,85.156998
3,Free,Paul Vincent,84.880997
4,"Eine Frau, Eine Wohnung, Ein Roman",Wilhelm Genazino,83.275002
5,Jade Peony,Wayson Choy,81.294998
6,Little Town On The Prairie,Laura Ingalls Wilder,27.118999
7,Hard Eight : A Stephanie Plum Novel (A Stephan...,Janet Evanovich,17.878000
8,Deck The Halls,Mary Higgins Clark,16.920000
9,Emily Of New Moon,L.M. Montgomery,16.857000


USER ID: 114368

Top Books Rated Highly by User:
  - [10/10] A Little Magic
  - [10/10] Thurston House
  - [10/10] A Time To Kill
  - [10/10] Carolina Moon
  - [10/10] Engaging The Enemy

Top 10 Recommended Books:


,Title,Author,Score
0,"Novocento, Un Monologo",Alessandro Baricco,35.594002
1,Oceano Mare,Alessandro Baricco,34.959000
2,Due Di Due (Bestsellers),Andrea De Carlo,34.673000
3,Jade Peony,Wayson Choy,33.562000
4,"Eine Frau, Eine Wohnung, Ein Roman",Wilhelm Genazino,28.399000
5,Free,Paul Vincent,28.399000
6,Rising Tides,Nora Roberts,18.158001
7,Hot Ice,Nora Roberts,16.112000
8,Brazen Virtue,Nora Roberts,15.744000
9,Temptation,Nora Roberts,14.965000


USER ID: 123883

Top Books Rated Highly by User:
  - [5/10] A Walk To Remember
  - [5/10] Sea Swept (Quinn Brothers (Paperback))
  - [5/10] Rising Tides
  - [5/10] Inner Harbor (Quinn Brothers (Paperback))
  - [5/10] Key Of Knowledge (Key Trilogy (Paperback))

Top 10 Recommended Books:


,Title,Author,Score
0,River'S End,Nora Roberts,3.664
1,Time And Again,Nora Roberts,2.321
2,Hush,Anne Frasier,1.541
3,Wild Orchids : A Novel,Jude Deveraux,1.381
4,Johnny Angel,Danielle Steel,1.346
5,An Accidental Woman,Barbara Delinsky,1.247
6,Birthright,Nora Roberts,1.241
7,Born In Fire,Nora Roberts,1.175
8,Reunion In Death,J. D. Robb,1.150
9,The Rescue,Nicholas Sparks,0.892


In [9]:
import shutil
from google.colab import files

shutil.make_archive("silver_gold_layers", "zip", ".", "silver")
!zip -r silver_gold_layers.zip gold/
files.download("silver_gold_layers.zip")

  adding: gold/ (stored 0%)
  adding: gold/user_book_matrix.npz (deflated 1%)
  adding: gold/book_lookup.parquet (deflated 12%)
  adding: gold/book_mapping.parquet (deflated 46%)
  adding: gold/user_mapping.parquet (deflated 51%)
  adding: gold/book_aggregates.parquet (deflated 16%)
  adding: gold/user_aggregates.parquet (deflated 27%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>